## step 1 - import all required softwares

In [ ]:

from langchain_community.document_loaders import PyPDFLoader   # to load the pdf
from langchain_text_splitters import RecursiveCharacterTextSplitter    # to convert pages to chunks
from langchain_huggingface import HuggingFaceEmbeddings    # to convert text to vector form
import faiss      # to find len of chunks
import os
from langchain_community.vectorstores import FAISS  # to ceate vector database
from openai import OpenAI

## step 2 - load tcs pdf file and load all the pages

In [ ]:
loader = PyPDFLoader('tcs.pdf')
mydocuments = loader.load()

In [ ]:
mydocuments[0].page_content[:1000]

## step 3 - convert documents to chunks


In [ ]:

text_splitter = RecursiveCharacterTextSplitter(chunk_size=2500,
			chunk_overlap=500)
mychunks = text_splitter.split_documents(mydocuments)

## step 4 - take vector model

In [11]:
model = HuggingFaceEmbeddings(model_name = 'sentence-transformers/all-mpnet-base-v2')


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 310.93it/s]


## step 5 - create faiss db and pass documents and vector model

In [12]:
vector_store = FAISS.from_documents(mychunks, model)

## step 6 - ask a question on tcs pdf get relevant chunks from faiss db


In [ ]:

question = "how is the tcs performing in terms of revenue?"

# Fix: Remove the standalone '3' and just use k=3
results = vector_store.similarity_search(question, k=3)


## step 7 - keep all 3 chunks results into one variable


In [18]:
our_context = ''
for res in results:
	our_context += res.page_content+'\n\n'


In [19]:
our_context

"of margins.\nThese were offset by investments in talent and capability building, \nand other investments for strengthening ecosystem partnerships and \nincreasing Go-to-Market pipeline.\nOperating Profit Trend (H crore)\nFY  2022\n48, 453\nFY  2024\n59 , 31 1\nFY  2023\n54, 2 37\nFY  2025\n62 , 1 65\nFY  2026\n66, 838\n2 5. 3% 2 5. 0 %\n2 4. 3%2 4. 6%2 4. 1 %\nO perati ng  P rof i t\nO perati ng  Mar g i n\nOver the past five fiscal years, TCS has steadily increased its Earnings \nPer Share (EPS), recording a compound annual growth rate of 10.3% . \nThis consistent upward trend demonstrates not only the company’s \nstrengthening profits but also its dedication to providing sustained value \nfor shareholders.\nEarnings per share (Amount in H)\n10. 3%\nC AG R\nFY  2022\n 1 0 3. 62  \nFY  2024\n 1 2 7. 74 \nFY  2023\n 1 1 5. 1 9  \nFY  2025\n 1 34. 1 9  \nFY  2026\n 1 45. 9 9  \nTCS has demonstrated a strong cash conversion ratio, which reflects its solid \nfinancial position and effecti

## step 8 - make a prompt

In [20]:

our_prompt = f"""
		I am going to ask a question based on my contxt. please answer the
		question based on my context only. if you dont find answer in my context
		then tell me you did nt find the answer, and dont give me answer from
                your knowledge.

		context = {our_context}
		question = {question}
	"""

## step 9 - give prompt to open ai llm and get response


In [ ]:
from google import genai


client = genai.Client(api_key=key)
response = client.interactions.create(model='gemini-3.6-flash', input = our_prompt)

print(response.output_text)

Based on the provided context, TCS's revenue performance is summarized as follows:

* **FY 2026 Growth:** TCS achieved a year-over-year revenue growth of **4.6%** in FY 2026.
* **Constant Currency Basis:** On a constant currency basis, revenue **declined by 2.4%**, largely due to one of its large transformation programs in India coming to an end.
* **5-Year Growth Trend (CAGR):** Over the past five financial years, TCS recorded a Compound Annual Growth Rate (CAGR) of **10.2%**.
* **Revenue Trend (in ₹ crore):**
  * **FY 2022:** 191,754
  * **FY 2023:** 225,458
  * **FY 2024:** 240,893
  * **FY 2025:** 255,324
  * **FY 2026:** 267,021


In [26]:
print(response)

status='completed' agent=None agent_config=None created='2026-09-19T16:50:40Z' environment=None environment_id=None errors=None generation_config=None id='v1_ChdZTDJ1YXJfVEhhSFRqdU1QcV8ySjhRVRIXWUwydWFyX1RIYUhUanVNUHFfMko4UVU' input=None labels=None model='gemini-3.6-flash' output_audio=None output_image=None output_text="Based on the provided context, TCS's revenue performance is summarized as follows:\n\n* **FY 2026 Growth:** TCS achieved a year-over-year revenue growth of **4.6%** in FY 2026.\n* **Constant Currency Basis:** On a constant currency basis, revenue **declined by 2.4%**, largely due to one of its large transformation programs in India coming to an end.\n* **5-Year Growth Trend (CAGR):** Over the past five financial years, TCS recorded a Compound Annual Growth Rate (CAGR) of **10.2%**.\n* **Revenue Trend (in ₹ crore):**\n  * **FY 2022:** 191,754\n  * **FY 2023:** 225,458\n  * **FY 2024:** 240,893\n  * **FY 2025:** 255,324\n  * **FY 2026:** 267,021" output_video=None previ